# 第 8 章 ナイーブベイズ

学習ループのない学習器です。単語を数え上げ、ラプラス平滑化を掛け、対数空間で確率を合成します。

対応する記事: [第 8 章 ナイーブベイズ（Kotlin Notebook の言語版）](../../../docs/article/grokking-machine-learning/kotlin/ch08.md)

実装本体: `apps/grokking-ml-kotlin/src/`

## セットアップ

実装本体をビルドした JAR を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

先に JAR を作っておいてください。

```bash
cd apps/grokking-ml-kotlin
./gradlew jar
```

IntelliJ IDEA の Kotlin Notebook プラグイン、または [Kotlin Jupyter カーネル](https://github.com/Kotlin/kotlin-jupyter) で開きます。

```bash
pip install kotlin-jupyter-kernel
jupyter lab notebooks/
```

In [1]:
@file:DependsOn("../build/libs/grokking-ml-kotlin-0.1.0.jar")

import ch08.*

## データセット

スパム 3 通、通常メール 5 通の小さなデータです。**学習は数え上げの 1 パスで終わります。**

In [2]:
val documents = listOf(
    "lottery sale", "lottery winning", "winning lottery sale", "sale today",
    "meeting tomorrow", "project meeting", "lunch meeting today", "project deadline",
)
val labels = listOf(1, 1, 1, 0, 0, 0, 0, 0)

val model = train(documents, labels)
println("スパム ${model.spamDocuments} 通 / 通常 ${model.hamDocuments} 通")
println("事前確率 %.4f".format(priorSpamProbability(model)))

スパム 3 通 / 通常 5 通
事前確率 0.3750


## ラプラス平滑化

`lottery` はスパム 3 件・通常 0 件です。**平滑化がないと確率 1.0 になり、他のどんな単語が来ても覆せません。** すべてのカウントに 1 を足すことで 0.8 に収まります。

未知語はちょうど 0.5 になり、**判定に寄与しません。**

In [3]:
println("%-10s %6s %6s %8s".format("単語", "スパム", "通常", "確率"))
listOf("lottery", "winning", "sale", "today", "meeting", "unseen").forEach { word ->
    println("%-10s %6d %6d %8.4f".format(word, model.spamWordCounts[word] ?: 0,
            model.hamWordCounts[word] ?: 0, wordSpamProbability(model, word)))
}

単語            スパム     通常       確率
lottery         3      0   0.8000


winning         2      0   0.7500
sale            2      1   0.6000


today           0      2   0.2500
meeting         0      3   0.2000


unseen          0      0   0.5000


## 証拠が積み重なる

スパム語が重なるほど確率が上がります。これが掛け算（実装上は対数の足し算）の効果です。

**空の文書は事前確率そのもの** を返します。単語による更新が 1 つも起きないためです。

In [4]:
listOf("", "project deadline", "sale today", "lottery",
       "lottery winning", "lottery winning sale").forEach { document ->
    val probability = predictProbability(model, document)
    val bar = "#".repeat((probability * 40).toInt())
    println("%-24s %.4f %s".format(if (document.isEmpty()) "(空)" else document, probability, bar))
}

(空)                      0.3750 ###############
project deadline         0.0909 ###


sale today               0.2308 #########


lottery                  0.7059 ############################


lottery winning          0.8780 ###################################


lottery winning sale     0.9153 ####################################


## 未知語は予測を変えない

平滑化により未知語の確率は 0.5 なので、含めても含めなくても結果は同じです。

In [5]:
println("lottery              %.6f".format(predictProbability(model, "lottery")))
println("lottery zzzz qqqq    %.6f".format(predictProbability(model, "lottery zzzz qqqq")))
println("正解率 %.2f".format(accuracy(model, documents, labels)))

lottery              0.705882
lottery zzzz qqqq    0.705882
正解率 1.00


## 試してみる: 学習データを足す

新しいメールを 1 通足すと、単語の確率がどう動くでしょうか。**学習ループがないので、足して数え直すだけです。**

In [6]:
val extended = train(documents + "meeting lottery", labels + 0)

println("%-10s %8s %8s".format("単語", "元", "追加後"))
listOf("lottery", "meeting").forEach { word ->
    println("%-10s %8.4f %8.4f".format(word, wordSpamProbability(model, word),
            wordSpamProbability(extended, word)))
}

単語                元      追加後
lottery      0.8000   0.6667
meeting      0.2000   0.1667
